<div align="right"><a href="S13-Designing-recursive-filters.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). Beachte bitte, dass zur Ausführung von MATLAB in Notebooks zunächst die [MATLAB API für Python](https://mathworks.com/help/matlab/matlab-engine-for-python.html) installiert werden muss. Die Installation wird [hier](https://mathworks.com/help/matlab/matlab_external/install-the-matlab-engine-for-python.html) beschrieben. <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/matlab/L13-Entwurf-rekursiver-Filter.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L12 Filterstrukturen](./L12-Filterstrukturen.ipynb) | [Aufgabe 13](./A13-Entwurf-rekursiver-Filter.ipynb) | [Inhalt / Übersicht](./00-MATLAB-Notebooks-Overview-DE.ipynb) | Weiter: [L14 Quantisierung der Filterkoeffizienten](./L14-Quantisierung-der-Filterkoeffizienten.ipynb)

</div>

---


## Aufgabe 4.2: Entwurf rekursiver Filter

Ziel dieser Aufgabe ist es, ein digitales Tiefpass-System basierend auf einem gegebenen Toleranzschema für den Amplitudengang zu entwerfen. Es stehen Python-Funktionen, z. B. `butter`, `cheby1` usw. aus `scipy.signal` zur Verfügung, um einen Tiefpassentwurf für kontinuierliche Zeitsysteme zu realisieren. Das in der Z-Ebene gegebene Toleranzschema muss in die S-Ebene transformiert werden. Die bekannten Algorithmen werden verwendet, um das gewünschte Filter in der S-Ebene zu entwerfen. Danach wird die Systemfunktion zurück in die Z-Ebene transformiert.

Für die Transformation von der Z- in die S-Ebene und umgekehrt wird die **bilineare Transformation** verwendet, welche stabile Systeme in der S-Ebene auf stabile Systeme in der Z-Ebene abbildet (Abschnitt 4.2.1). Bei Verwendung der bilinearen Transformation ist eine Vorverzerrung der $\Omega$-Achse in der Z-Ebene notwendig. Der Entwurf kann mittels der Standardverfahren gemäß Abschnitt 4.2.2 und 4.2.3 unter Verwendung einer Hilfsfunktion durchgeführt werden.

Für den Tiefpass-Entwurf stehen die in Python (`scipy.signal`) bereitgestellten Funktionen in der folgenden Tabelle zur Verfügung. Die Berechnung der erforderlichen Filterordnung und der normierten Grenzfrequenz\footnote{Alle Entwurfsfunktionen verwenden normierte Frequenzen zwischen 0 und 1. Die Normierung erfolgt in Bezug auf die halbe Abtastfrequenz. Die normierte Grenzfrequenz gibt die Frequenz mit $3\,\text{dB}$ Dämpfung an. Beispiel: Abtastfrequenz 1000 Hz, Grenzfrequenz 300 Hz, normierte Grenzfrequenz: `Wn=300/500=0.6`} kann mit folgenden Funktionen erfolgen:

| Filtertyp          | Bestimmung der Filterordnung                                        |
| ------------------ | ------------------------------------------------------------------ |
| Butterworth        | `buttord(Wp, Ws, Rp, Rs, fs=fs)`                                   |
| Tschebyscheff Typ I| `cheb1ord(Wp, Ws, Rp, Rs, fs=fs)`                                  |
| Tschebyscheff Typ II| `cheb2ord(Wp, Ws, Rp, Rs, fs=fs)`                                |
| Cauer              | `ellipord(Wp, Ws, Rp, Rs, fs=fs)`                                  |

Anschließend können die Koeffizienten mit den folgenden Entwurfsfunktionen berechnet werden:

| Filtertyp          | Entwurfsfunktion                                                    |
| ------------------ | ------------------------------------------------------------------ |
| Butterworth        | `butter(m, Wn, btype='low', output='ba', fs=fs)`                   |
| Tschebyscheff Typ I| `cheby1(m, Rp, Wn, btype='low', output='ba', fs=fs)`               |
| Tschebyscheff Typ II| `cheby2(m, Rs, Wn, btype='low', output='ba', fs=fs)`             |
| Cauer              | `ellip(m, Rp, Rs, Wn, btype='low', output='ba', fs=fs)`            |

Die Bedeutung der Parameter ist:

| Parameter          | Bedeutung                                                           |
| ------------------ | ------------------------------------------------------------------- |
| `b, a`             | Vektoren mit den Filterkoeffizienten                                |
| `m`                | Ordnung des Filters                                                 |
| `Wn`               | Normierte Grenzfrequenz des Tiefpasses                              |
| `Wp`               | Normierte Passbandfrequenz des Tiefpasses                           |
| `Ws`               | Normierte Sperrbandfrequenz des Tiefpasses                          |
| `Rp`               | Welligkeit im Passband (dB)                                         |
| `Rs`               | Dämpfung im Sperrband (dB)                                          |

Bestimme die minimale Filterordnung für jeden Filtertyp und entwirf die Tiefpass-Filter für das folgende Toleranzschema:

- $f_\text{D} = 1\,\text{kHz}$
- $f_\text{S} = 1{,}4\,\text{kHz}$
- $R_\text{P} = 0{,}5\,\text{dB}$
- $R_\text{S} = 30\,\text{dB}$
- $f_\text{A} = 8\,\text{kHz}$

![Toleranzschema](./pngs/toleranz.eps)

Vergleiche den Amplitudengang und das Nullstellen-Pol-Diagramm der verschiedenen Filter miteinander.

In [ ]:

# BEGIN SOLUTION

fD = 1000;    % Durchlassfrequenz (Hz)
fS = 1400;    % Sperrfrequenz (Hz)
Rp = 0.5;     % Welligkeit im Durchlassbereich (dB)
Rs = 30;      % Dämpfung im Sperrbereich (dB)
fA = 8000;    % Abtastrate (Hz)

% Umrechnung in normalisierte Frequenzen (bezogen auf Nyquist-Frequenz)
Wp = 2 * fD / fA; 
Ws = 2 * fS / fA;

for typ = 1:4
    % Auswahl des Filtertyps
    if typ == 1
        [n, Wn] = buttord(Wp, Ws, Rp, Rs);    % Butterworth-Filter
        [b, a] = butter(n, Wn);
        FType = 'Butterworth';
    elseif typ == 2
        [n, Wn] = cheb1ord(Wp, Ws, Rp, Rs);   % Tschebyschew Typ I
        [b, a] = cheby1(n, Rp, Wn);
        FType = 'Chebyshev Type I';
    elseif typ == 3
        [n, Wn] = cheb2ord(Wp, Ws, Rp, Rs);   % Tschebyschew Typ II
        [b, a] = cheby2(n, Rs, Wn);
        FType = 'Chebyshev Type II';
    elseif typ == 4
        [n, Wn] = ellipord(Wp, Ws, Rp, Rs);   % Cauer-/Elliptischer Filter
        [b, a] = ellip(n, Rp, Rs, Wn);
        FType = 'Cauer';
    else
        error('Unbekannter Filtertyp...');
    end

    % Berechnung des Frequenzgangs
    [H, W] = freqz(b, a, 512);
    
    % === Grafische Ausgabe === %
    TitleString = sprintf('Filtertyp: %s  Ordnung: %d', FType, n);
    
    % Amplitudengang
    figure;
    plot(W/pi, abs(H));
    grid;
    title(TitleString);
    xlabel('\Omega/\pi');
    ylabel('|H(exp(j\Omega))|');
    axis([0 1 0 1.2]);
    
    % Toleranzband einzeichnen
    xline = [Wp Wp];
    yline = [0 10.^(-Rp/20)];           % untere Grenze Passband
    line(xline, yline, 'linestyle', '--');
    
    xline = [0 Ws];
    yline = [1 1];                      % obere Grenze Passband
    line(xline, yline, 'linestyle', '--');
    
    xline = [0 Wp];
    yline = [10.^(-Rp/20) 10.^(-Rp/20)]; % Toleranzgrenze im Durchlassbereich
    line(xline, yline, 'linestyle', '--');
    
    xline = [Ws Ws];
    yline = [10.^(-Rs/20) 1];           % Übergang zur Sperrfrequenz
    line(xline, yline, 'linestyle', '--');
    
    xline = [Ws 1];
    yline = [10.^(-Rs/20) 10.^(-Rs/20)]; % Sperrbereich (max. zulässige Amplitude)
    line(xline, yline, 'linestyle', '--');
    
    % Nullstellen-Pol-Diagramm in der z-Ebene
    figure;
    zplane(b, a);
    title(TitleString);
    xlabel('Reeller Teil');
    ylabel('Imaginärteil');
end

# END SOLUTION